<a href="https://colab.research.google.com/github/Muen1/multilingual-health-qa-africa/blob/main/notebook/03_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os

BASE_DIR = '/content/drive/MyDrive/multilingual_health_qa'
DATA_DIR = f'{BASE_DIR}/data'
OUT_DIR  = f'{BASE_DIR}/outputs'
PLOT_DIR = f'{BASE_DIR}/plots'

for d in [OUT_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted.")
print("Data files:", os.listdir(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
Data files: ['Train.csv', 'Val.csv', 'Test.csv', 'SampleSubmission.csv', 'train_clean.csv', 'val_clean.csv', 'test_clean.csv']


In [2]:
# Install dependencies
!pip install -q transformers==4.40.0 datasets==2.19.0 peft==0.10.0 \
             accelerate==0.29.3 \
             rouge-score sentencepiece protobuf evaluate

print("Installation complete.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
Installation complete.


In [3]:
# imports
import gc
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import evaluate as hf_evaluate
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4


In [4]:
#Load cleaned data
train_raw = pd.read_csv(f'{DATA_DIR}/train_clean.csv')
val_raw   = pd.read_csv(f'{DATA_DIR}/val_clean.csv')
test_raw  = pd.read_csv(f'{DATA_DIR}/test_clean.csv')

q_col    = "input"
a_col    = "output"
lang_col = "subset"
id_col   = "ID"

def build_prompt_v1(q, l):
    return f"Language: {l}\nQuestion: {q}\nAnswer:"

def build_prompt_v2(q, l):
    return (f"You are a health assistant. Answer the following health question "
            f"in {l}.\nQuestion: {q}\nAnswer:")

# Keep raw frames untouched — each experiment rebuilds prompts from these
print(f"Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}")
print("Sample question:", train_raw[q_col].iloc[0])

Train: 29814, Val: 6686, Test: 2618
Sample question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.


In [5]:
# all six experiments
configs = [
    dict(
        EXPERIMENT_NUM=1, EXPERIMENT_NAME="exp01_mt5small_zeroshot",
        MODEL_NAME="google/mt5-small", USE_LORA=False,
        LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.1,
        LEARNING_RATE=3e-4, NUM_EPOCHS=3, BATCH_SIZE=8,
        MAX_INPUT_LEN=256, MAX_TARGET_LEN=256,
        SKIP_TRAINING=True, PROMPT_VERSION="v1",
    ),
    dict(
        EXPERIMENT_NUM=2, EXPERIMENT_NAME="exp02_mt5small_finetuned",
        MODEL_NAME="google/mt5-small", USE_LORA=True,
        LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.1,
        LEARNING_RATE=3e-4, NUM_EPOCHS=3, BATCH_SIZE=8,
        MAX_INPUT_LEN=256, MAX_TARGET_LEN=256,
        SKIP_TRAINING=False, PROMPT_VERSION="v1",
    ),
    dict(
        EXPERIMENT_NUM=3, EXPERIMENT_NAME="exp03_flanT5base_r16",
        MODEL_NAME="google/flan-t5-base", USE_LORA=True,
        LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.1,
        LEARNING_RATE=3e-4, NUM_EPOCHS=3, BATCH_SIZE=8,
        MAX_INPUT_LEN=256, MAX_TARGET_LEN=256,
        SKIP_TRAINING=False, PROMPT_VERSION="v1",
    ),
    dict(
        EXPERIMENT_NUM=4, EXPERIMENT_NAME="exp04_flanT5base_r32",
        MODEL_NAME="google/flan-t5-base", USE_LORA=True,
        LORA_R=32, LORA_ALPHA=64, LORA_DROPOUT=0.1,
        LEARNING_RATE=3e-4, NUM_EPOCHS=3, BATCH_SIZE=8,
        MAX_INPUT_LEN=256, MAX_TARGET_LEN=256,
        SKIP_TRAINING=False, PROMPT_VERSION="v1",
    ),
    dict(
        EXPERIMENT_NUM=5, EXPERIMENT_NAME="exp05_flanT5base_promptv2",
        MODEL_NAME="google/flan-t5-base", USE_LORA=True,
        LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.1,
        LEARNING_RATE=3e-4, NUM_EPOCHS=3, BATCH_SIZE=8,
        MAX_INPUT_LEN=256, MAX_TARGET_LEN=256,
        SKIP_TRAINING=False, PROMPT_VERSION="v2",
    ),
    dict(
        EXPERIMENT_NUM=6, EXPERIMENT_NAME="exp06_flanT5large_r16",
        MODEL_NAME="google/flan-t5-large", USE_LORA=True,
        LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.1,
        LEARNING_RATE=3e-4, NUM_EPOCHS=3, BATCH_SIZE=4,
        MAX_INPUT_LEN=256, MAX_TARGET_LEN=256,
        SKIP_TRAINING=False, PROMPT_VERSION="v1",
    ),
]

for c in configs:
    print(c["EXPERIMENT_NUM"], c["EXPERIMENT_NAME"])

1 exp01_mt5small_zeroshot
2 exp02_mt5small_finetuned
3 exp03_flanT5base_r16
4 exp04_flanT5base_r32
5 exp05_flanT5base_promptv2
6 exp06_flanT5large_r16


In [6]:
# ROUGE metric (loaded once, used by every experiment)
rouge = hf_evaluate.load("rouge")

def make_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds  = [p.strip() for p in decoded_preds]
        decoded_labels = [l.strip() for l in decoded_labels]
        result = rouge.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            use_stemmer=True
        )
        return {
            "rouge1": round(result["rouge1"], 4),
            "rougeL": round(result["rougeL"], 4)
        }
    return compute_metrics

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
# the experiment function
def run_experiment(cfg):
    EXPERIMENT_NUM  = cfg["EXPERIMENT_NUM"]
    EXPERIMENT_NAME = cfg["EXPERIMENT_NAME"]
    MODEL_NAME      = cfg["MODEL_NAME"]
    USE_LORA        = cfg["USE_LORA"]
    LORA_R          = cfg["LORA_R"]
    LORA_ALPHA      = cfg["LORA_ALPHA"]
    LORA_DROPOUT    = cfg["LORA_DROPOUT"]
    LEARNING_RATE   = cfg["LEARNING_RATE"]
    NUM_EPOCHS      = cfg["NUM_EPOCHS"]
    BATCH_SIZE      = cfg["BATCH_SIZE"]
    MAX_INPUT_LEN   = cfg["MAX_INPUT_LEN"]
    MAX_TARGET_LEN  = cfg["MAX_TARGET_LEN"]
    SKIP_TRAINING   = cfg["SKIP_TRAINING"]
    PROMPT_VERSION  = cfg["PROMPT_VERSION"]

    print("=" * 70)
    print(f"Running : {EXPERIMENT_NAME}")
    print(f"Model   : {MODEL_NAME}")
    print(f"LoRA    : {USE_LORA} (r={LORA_R})")
    print(f"LR      : {LEARNING_RATE}")
    print(f"Epochs  : {NUM_EPOCHS}")
    print(f"Prompt  : {PROMPT_VERSION}")
    print(f"Skip training (zero-shot) : {SKIP_TRAINING}")
    print("=" * 70)

    # --- Build prompts fresh from raw data (no cross-experiment leakage) ---
    build_prompt = build_prompt_v2 if PROMPT_VERSION == "v2" else build_prompt_v1

    train = train_raw.copy()
    val   = val_raw.copy()
    test  = test_raw.copy()

    train['input_text']  = train.apply(lambda r: build_prompt(r[q_col], r[lang_col]), axis=1)
    train['target_text'] = train[a_col].astype(str)
    val['input_text']    = val.apply(lambda r: build_prompt(r[q_col], r[lang_col]), axis=1)
    val['target_text']   = val[a_col].astype(str)
    test['input_text']   = test.apply(lambda r: build_prompt(r[q_col], r[lang_col]), axis=1)

    print("Sample input:", train['input_text'].iloc[0])

    # --- Load fresh tokenizer + model every time ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    print(f"Total parameters: {model.num_parameters():,}")

    if USE_LORA:
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            target_modules=["q", "v"],
            lora_dropout=LORA_DROPOUT,
            bias="none"
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
    else:
        total = sum(p.numel() for p in model.parameters())
        print(f"No LoRA — full model. Trainable parameters: {total:,}")

    # --- Tokenize (skip if zero-shot) ---
    trainer = None
    if not SKIP_TRAINING:
        def preprocess(examples):
            inputs = tokenizer(
                examples['input_text'],
                max_length=MAX_INPUT_LEN,
                truncation=True,
                padding='max_length'
            )
            targets = tokenizer(
                examples['target_text'],
                max_length=MAX_TARGET_LEN,
                truncation=True,
                padding='max_length'
            )
            labels = [
                [(l if l != tokenizer.pad_token_id else -100) for l in label]
                for label in targets['input_ids']
            ]
            inputs['labels'] = labels
            return inputs

        train_ds = Dataset.from_pandas(
            train[['input_text', 'target_text']].reset_index(drop=True))
        val_ds   = Dataset.from_pandas(
            val[['input_text', 'target_text']].reset_index(drop=True))

        train_tok = train_ds.map(
            preprocess, batched=True,
            remove_columns=['input_text', 'target_text'])
        val_tok   = val_ds.map(
            preprocess, batched=True,
            remove_columns=['input_text', 'target_text'])
        print("Tokenization complete.")
        print(f"Train tokens: {len(train_tok)}, Val tokens: {len(val_tok)}")

        # --- Train ---
        training_args = Seq2SeqTrainingArguments(
            output_dir=f"{OUT_DIR}/checkpoints_{EXPERIMENT_NAME}",
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=4,
            warmup_ratio=0.1,
            learning_rate=LEARNING_RATE,
            weight_decay=0.01,
            fp16=True,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            predict_with_generate=True,
            generation_max_length=MAX_TARGET_LEN,
            logging_steps=50,
            report_to="none",
            push_to_hub=False
        )
        data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
        trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=train_tok,
            eval_dataset=val_tok,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=make_compute_metrics(tokenizer),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        est_min = len(train_tok) * NUM_EPOCHS / (BATCH_SIZE * 4) / 60
        print(f"Estimated training time: ~{est_min:.0f} minutes")
        print("Starting training...")
        train_result = trainer.train()
        print("Training complete.")
        print("Metrics:", train_result.metrics)
    else:
        print("Zero-shot experiment — skipping tokenization and training.")

    # --- Plot learning curves ---
    if not SKIP_TRAINING and trainer is not None:
        log_history = trainer.state.log_history

        train_steps  = [x['step']      for x in log_history if 'loss' in x and 'eval_loss' not in x]
        train_losses = [x['loss']      for x in log_history if 'loss' in x and 'eval_loss' not in x]
        eval_epochs  = [x['epoch']     for x in log_history if 'eval_loss' in x]
        eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]
        eval_rouge1  = [x.get('eval_rouge1', 0) for x in log_history if 'eval_loss' in x]
        eval_rougeL  = [x.get('eval_rougeL', 0) for x in log_history if 'eval_loss' in x]

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Learning Curves — {EXPERIMENT_NAME}', fontsize=14, fontweight='bold')

        axes[0].plot(train_steps, train_losses, color='royalblue', linewidth=1.5)
        axes[0].set_title('Training Loss')
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('Loss')
        axes[0].grid(True, alpha=0.4)

        axes[1].plot(eval_epochs, eval_losses, color='darkorange', marker='o', linewidth=2)
        axes[1].set_title('Validation Loss per Epoch')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Eval Loss')
        axes[1].grid(True, alpha=0.4)

        axes[2].plot(eval_epochs, eval_rouge1, color='green',  marker='o', linewidth=2, label='ROUGE-1')
        axes[2].plot(eval_epochs, eval_rougeL, color='purple', marker='s', linewidth=2, label='ROUGE-L')
        axes[2].set_title('ROUGE Scores per Epoch')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('F1 Score')
        axes[2].legend()
        axes[2].grid(True, alpha=0.4)

        plt.tight_layout()
        save_path = f'{PLOT_DIR}/learning_curves_{EXPERIMENT_NAME}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: {save_path}")
    else:
        print("No learning curves for zero-shot experiment.")

    # --- Save fine-tuned model ---
    if not SKIP_TRAINING:
        model_save_path = f'{OUT_DIR}/model_{EXPERIMENT_NAME}'
        os.makedirs(model_save_path, exist_ok=True)
        model.save_pretrained(model_save_path)
        tokenizer.save_pretrained(model_save_path)
        print(f"Model saved to: {model_save_path}")
    else:
        print("Zero-shot — no model to save.")

    # --- Inference on test set ---
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()

    def generate_answers(texts, batch_size=16, num_beams=4):
        all_answers = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Generating answers"):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors='pt',
                max_length=MAX_INPUT_LEN,
                truncation=True,
                padding=True
            )
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_TARGET_LEN,
                    num_beams=num_beams,
                    early_stopping=True,
                    no_repeat_ngram_size=3,
                    length_penalty=1.0
                )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_answers.extend(decoded)
        return all_answers

    print("Generating predictions on test set...")
    test_answers = generate_answers(test['input_text'].tolist(), num_beams=4)
    test['predicted_answer'] = test_answers

    predictions_path = f'{DATA_DIR}/predictions_{EXPERIMENT_NAME}.csv'
    test.to_csv(predictions_path, index=False)
    print(f"Saved predictions: {predictions_path}")
    print("\nSample predictions:")
    for i in range(3):
        print(f"\nQ: {test['input_text'].iloc[i][:100]}")
        print(f"A: {test['predicted_answer'].iloc[i][:150]}")

    # --- Cleanup to free GPU memory before next experiment ---
    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n Finished {EXPERIMENT_NAME}\n")

In [8]:
# Run a single experiment by index: 0=exp1, 1=exp2, 2=exp3, 3=exp4, 4=exp5, 5=exp6
run_experiment(configs[0])

Running : exp01_mt5small_zeroshot
Model   : google/mt5-small
LoRA    : False (r=16)
LR      : 0.0003
Epochs  : 3
Prompt  : v1
Skip training (zero-shot) : True
Sample input: Language: Aka_Gha
Question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.
Answer:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Total parameters: 300,176,768
No LoRA — full model. Trainable parameters: 300,176,768
Zero-shot experiment — skipping tokenization and training.
No learning curves for zero-shot experiment.
Zero-shot — no model to save.
Generating predictions on test set...


Generating answers: 100%|██████████| 164/164 [00:39<00:00,  4.13it/s]


Saved predictions: /content/drive/MyDrive/multilingual_health_qa/data/predictions_exp01_mt5small_zeroshot.csv

Sample predictions:

Q: Language: Aka_Gha
Question: Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ah
A: <extra_id_0>.

Q: Language: Aka_Gha
Question: Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ
A: <extra_id_0>?e蝾

Q: Language: Aka_Gha
Question: Akwan bɛn na mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi aba ɛfa nnipa 
A: <extra_id_0>.

 Finished exp01_mt5small_zeroshot

